In [6]:
import sys
import os

PROJECT_ROOT = os.path.dirname(os.getcwd())
HPC_DIR = os.path.join(PROJECT_ROOT, "hpc")
for p in (PROJECT_ROOT, HPC_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

In [7]:
import numpy as np
from qiskit_algorithms.optimizers import COBYLA

from parallel_training import TrainingConfig
from parallel_losses_circle_from_Eff import load_circle_data
from src.shot_simulation import create_sampler
from src.feature_map import TrainableQuantumFeatureMap
from src.kernel_estimate import KernelMatrix
from src.classifier import ClassicalClassifier

In [8]:
config = TrainingConfig(
    run_id=1,
    depth=2,
    type_ansatz="EfficientSU2",
    optimizer_name="COBYLA",
    maxiter=200,
    warmup_iterations=5,
    shots=1024,
    n_train=100,
    n_val=50,
    n_test=100,
    results_dir="../results_losses_circle_from_Eff_smoke",
    init_theta_source_dir="../results_losses_circle_Eff",
)
config.validate()

X_train, y_train, X_val, y_val, X_test, y_test = load_circle_data(config)

print("X_train", X_train.shape, "y_train", y_train.shape, "unique labels:", np.unique(y_train))
print("X_val  ", X_val.shape, "  y_val  ", y_val.shape, "  unique labels:", np.unique(y_val))
print("X_test ", X_test.shape, " y_test ", y_test.shape, " unique labels:", np.unique(y_test))

X_train (100, 2) y_train (100,) unique labels: [0 1]
X_val   (50, 2)   y_val   (50,)   unique labels: [0 1]
X_test  (100, 2)  y_test  (100,)  unique labels: [0 1]


In [ ]:
init_theta_file = (
    f"{config.init_theta_source_dir}/shared_init_theta_"
    f"depth{config.depth}_ansatz{config.type_ansatz}_run{config.run_id}.npy"
)
init_theta = np.load(init_theta_file)
print("init_theta shape:", init_theta.shape)

sampler = create_sampler(shots=config.shots)

tqfm = TrainableQuantumFeatureMap(
    depth=config.depth,
    type_ansatz=config.type_ansatz,
    type_loss="hilbert_schmidt",
    warmup_iterations=config.warmup_iterations,
    shots=config.shots,
    sampler=sampler,
)

optimizer = COBYLA(maxiter=config.maxiter)
tqfm.fit(X_train, y_train, X_val, y_val, optimizer=optimizer, init_theta=init_theta.copy())

print("optimal_value:", tqfm.optimal_value)
print("best_val_acc:", tqfm.best_val_acc)

init_theta shape: (24,)
Number of qubits: 2, Number of classes: 2
Shot-based simulation: 1024 shots
Total theta params for EfficientSU2: 24
Using loss strategy: HilbertSchmidtLoss


C:\Users\Nguyen Dinh Quyen\AppData\Local\Temp\ipykernel_10340\2348759258.py:8: DeprecationWarning: Sampler has been deprecated as of Aer 0.15, please use SamplerV2 instead.
  sampler = create_sampler(shots=config.shots)


Iteration 1: loss = 0.912510, best_val_acc = 0.0000
Iteration 2: loss = 0.916810, best_val_acc = 0.0000
Iteration 3: loss = 0.941972, best_val_acc = 0.0000
Iteration 4: loss = 0.873329, best_val_acc = 0.0000
Iteration 5: loss = 0.928715, best_val_acc = 0.0000


In [10]:
theta_for_eval = tqfm.best_params if tqfm.best_params is not None else tqfm.optimal_params

X_train_val = np.concatenate([X_train, X_val], axis=0)
y_train_val = np.concatenate([np.ravel(y_train), np.ravel(y_val)], axis=0).astype(int)

kernel_train_val = KernelMatrix.compute_kernel_matrix_with_inner_products(
    X_train_val, X_train_val, theta_for_eval, tqfm.circuit, sampler=sampler, shots=config.shots,
)
kernel_test = KernelMatrix.compute_kernel_matrix_with_inner_products(
    X_test, X_train_val, theta_for_eval, tqfm.circuit, sampler=sampler, shots=config.shots,
)

_, test_acc, _, best_c = ClassicalClassifier.evaluate_model(
    kernel_train_val, kernel_test, y_train_val, y_test
)
print("Test accuracy:", test_acc, "| Best C:", best_c)

Test accuracy: 0.95 | Best C: 10.0
